# Unsupervised NLP, Vision & Multimodal Clustering

End-to-end unsupervised learning pipeline covering three domains:

| Part | Domain | Task |
|------|---------|------|
| 1 | Text (Steam Reviews) | Discover review length structure without labels |
| 2 | Text (Steam Reviews) | Infer game genres from aggregated positive reviews |
| 3 | Vision (TF-Flowers) | Cluster flower images using VGG16 transfer features |
| 4 | Multimodal (Pokémon) | CLIP-based image–text retrieval and type classification |
| 5 | SQL Analytics | Relational analysis of the Steam dataset |

**Pipeline modules (Parts 1–3):**

| Stage | Options |
|---|---|
| Representation | TF-IDF · MiniLM · VGG16 features |
| Dimensionality Reduction | None · SVD(50) · UMAP(50) · Autoencoder(50) |
| Clustering | K-Means · Agglomerative · HDBSCAN |

## 0. Configuration

Set `USE_COLAB = True` when running in Google Colab (auto-mounts Drive).  
Set `USE_COLAB = False` for local Jupyter / VS Code (reads from `LOCAL_*` paths).

In [ ]:
# ── Environment toggle ─────────────────────────────────────────────────────────
USE_COLAB = True

# Steam reviews (Parts 1, 2, 5)
COLAB_REVIEWS_PATH  = "/content/drive/MyDrive/steam_reviews/main.csv"
LOCAL_REVIEWS_PATH  = "data/main.csv"

# Held-out game (Part 2 – Task 3)
COLAB_HELDOUT_PATH  = "/content/drive/MyDrive/steam_reviews/heldout.csv"
LOCAL_HELDOUT_PATH  = "data/heldout.csv"

# VGG features cache (Part 3 – auto-downloaded if absent)
VGG_FEATURES_PATH   = "flowers_features_and_labels.npz"

# Pokémon CSV (Part 4 – fetched from GitHub)
POKEMON_CSV_URL     = "https://raw.githubusercontent.com/lgreski/pokemonData/master/Pokemon.csv"

# SQLite database (Part 5 – created automatically)
DB_PATH             = "steam_reviews.db"
# ──────────────────────────────────────────────────────────────────────────────

RANDOM_STATE = 42

## 1. Setup & Imports

In [ ]:
!pip install sentence-transformers umap-learn hdbscan tensorflow seaborn
!pip install torch torchvision transformers datasets
!pip install git+https://github.com/openai/CLIP.git kagglehub plotly
!pip install scipy kagglehub

In [ ]:
import re
import os
import random
import sqlite3
import tarfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from glob import glob
from scipy.special import softmax

import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize

from PIL import Image
import requests

import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, TensorDataset

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

import clip
import kagglehub

from sklearn.base import TransformerMixin
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.metrics import (
    homogeneity_score, completeness_score, v_measure_score,
    adjusted_rand_score, adjusted_mutual_info_score,
)
from sklearn.model_selection import train_test_split

from umap import UMAP
import hdbscan
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
)

for resource in ["wordnet", "averaged_perceptron_tagger", "punkt_tab", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource, quiet=True)

print("All imports successful.")


## 2. Shared Utilities

In [ ]:
# ── Data loading ───────────────────────────────────────────────────────────────

def load_csv(colab_path: str, local_path: str, label: str = "") -> pd.DataFrame:
    """
    Mount Google Drive (if USE_COLAB) and load a CSV file.

    Parameters
    ----------
    colab_path : str  — absolute path inside mounted Drive
    local_path : str  — relative or absolute local path
    label      : str  — human-readable name for the dataset (for logging)
    """
    if USE_COLAB:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        path = colab_path
    else:
        path = local_path
    df = pd.read_csv(path)
    print(f"Loaded {label}: {len(df):,} rows from '{path}'")
    return df


# ── Text preprocessing ─────────────────────────────────────────────────────────

_lemmatizer = WordNetLemmatizer()


def _get_wordnet_pos(word: str) -> str:
    """Map an NLTK POS tag to the corresponding WordNet POS constant."""
    tag = pos_tag([word])[0][1][0].upper()
    return {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}.get(tag, wordnet.NOUN)


def clean_text(text: str) -> str:
    """Strip HTML, URLs, and noise characters; normalise whitespace."""
    text = str(text).strip("'")
    text = re.sub(r"^https?://.*[\r\n]*", "", text, flags=re.MULTILINE)
    text = re.sub(r"<br />|<.*?>", " ", text)
    text = re.sub(r"&quot;|&#39;", '"', text)
    text = re.sub(r"&amp;", "and", text)
    text = re.sub(r"[\n\r]", " ", text)
    text = re.sub(r" u ", " you ", text)
    text = re.sub(r"[`,']", "", text)
    text = re.sub(r"\s*[-\u2013\u2014]\s*", " ", text)
    text = re.sub(r"(!)+", "!", text)
    text = re.sub(r"(\?)+", "?", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    return re.sub(r" +", " ", text).strip()


def lemmatize_text(text: str) -> str:
    """Tokenise and POS-aware lemmatise a cleaned string."""
    tokens = word_tokenize(text)
    return " ".join(_lemmatizer.lemmatize(w, _get_wordnet_pos(w)) for w in tokens)


def preprocess(text: str) -> str:
    """Full text preprocessing pipeline: clean → lemmatise."""
    return lemmatize_text(clean_text(text))


# ── Clustering benchmark ───────────────────────────────────────────────────────

def run_clustering_benchmark(
    representations: dict,
    ground_truth: list,
    n_clusters: int = 2,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Evaluate every (representation × DR × clustering) combination and return
    a DataFrame of clustering agreement metrics.

    Parameters
    ----------
    representations : dict  — {name: sparse or dense matrix}
    ground_truth    : list  — true labels used only for metric computation
    n_clusters      : int   — target clusters for K-Means / Agglomerative
    random_state    : int

    Returns
    -------
    pd.DataFrame — one row per pipeline, columns for each metric
    """
    dr_configs = {
        "None" : None,
        "SVD"  : TruncatedSVD(n_components=50, random_state=random_state),
        "UMAP" : UMAP(n_components=50, random_state=random_state, n_neighbors=15, min_dist=0.1),
    }
    cluster_configs = {
        "K-Means"      : KMeans(n_clusters=n_clusters, random_state=random_state, n_init="auto"),
        "Agglomerative": AgglomerativeClustering(n_clusters=n_clusters, linkage="ward"),
    }

    records = []
    for rep_name, data in representations.items():
        for dr_name, dr_model in dr_configs.items():
            try:
                if dr_name == "None":
                    X = data.toarray() if hasattr(data, "toarray") else data
                elif dr_name == "SVD":
                    X = dr_model.fit_transform(data)
                else:  # UMAP — SVD pre-reduction for sparse TF-IDF
                    pre = TruncatedSVD(n_components=50, random_state=random_state).fit_transform(data) \
                          if rep_name == "TF-IDF" else data
                    X = dr_model.fit_transform(pre)
            except Exception as e:
                print(f"  DR failed [{rep_name}+{dr_name}]: {e}")
                continue

            for cl_name, cl_model in cluster_configs.items():
                if cl_name == "Agglomerative" and dr_name == "None" and rep_name == "TF-IDF":
                    continue  # Agglomerative requires dense input
                try:
                    labels = cl_model.fit_predict(X)
                    records.append({
                        "Representation": rep_name, "DR": dr_name, "Clustering": cl_name,
                        "Homogeneity" : homogeneity_score(ground_truth, labels),
                        "Completeness": completeness_score(ground_truth, labels),
                        "V-Measure"   : v_measure_score(ground_truth, labels),
                        "ARI"         : adjusted_rand_score(ground_truth, labels),
                        "AMI"         : adjusted_mutual_info_score(ground_truth, labels),
                    })
                except Exception as e:
                    print(f"  Cluster failed [{rep_name}+{dr_name}+{cl_name}]: {e}")

    return pd.DataFrame(records)


print("Utilities defined.")

---
# Part 1 — Review Length Discovery

**Question:** Can unsupervised clustering recover short vs. long review structure from text representations alone, without ever seeing length labels?

**Pseudo-label construction** (evaluation only — hidden from clustering):
- **Short** = bottom 25% by token count  
- **Long**  = top 25% by token count  
- Middle 50% discarded for clean class separation

### 1.1 — Load & Preprocess

In [ ]:
df = load_csv(COLAB_REVIEWS_PATH, LOCAL_REVIEWS_PATH, "Steam reviews")

print("Applying preprocessing pipeline (clean → lemmatise)...")
df["processed_review_text"] = df["review_text"].astype(str).apply(preprocess)
df["review_length"]         = df["review_text"].astype(str).apply(lambda x: len(word_tokenize(x)))
print("Done.")

### 1.2 — Build Short / Long Subsets

In [ ]:
q25 = df["review_length"].quantile(0.25)
q75 = df["review_length"].quantile(0.75)

short_df = df[df["review_length"] <= q25]
long_df  = df[df["review_length"] >= q75]
retained = pd.concat([short_df, long_df])

print(f"Reviews retained : {len(retained):,} / {len(df):,}")
print(f"Avg length — Short : {short_df['review_length'].mean():.1f} tokens")
print(f"Avg length — Long  : {long_df['review_length'].mean():.1f} tokens")

### 1.3 — Build Representations (TF-IDF & MiniLM)

In [ ]:
processed_texts = retained["processed_review_text"].astype(str).tolist()
raw_texts       = retained["review_text"].tolist()

# TF-IDF: unigrams, min_df=3, English stopwords
tfidf_vec    = TfidfVectorizer(min_df=3, stop_words="english", ngram_range=(1, 1))
tfidf_matrix = tfidf_vec.fit_transform(processed_texts)
print(f"TF-IDF matrix     : {tfidf_matrix.shape}  (sparse — most entries zero)")

# MiniLM: dense contextual sentence embeddings
minilm        = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
minilm_embeds = minilm.encode(raw_texts, show_progress_bar=True)
print(f"MiniLM embeddings : {minilm_embeds.shape}  (dense — every dimension populated)")

### 1.4 — Clustering Benchmark

In [ ]:
gt_labels_p1 = retained["review_length"].apply(lambda x: "Short" if x <= q25 else "Long").tolist()

results_p1 = run_clustering_benchmark(
    representations={"TF-IDF": tfidf_matrix, "MiniLM": minilm_embeds},
    ground_truth=gt_labels_p1,
)

print("\nPart 1 — Results (sorted by V-Measure):")
print(results_p1.sort_values("V-Measure", ascending=False).to_string(index=False))

### 1.5 — Visualisation: Ground Truth vs. Cluster Assignments

In [ ]:
def plot_clustering_comparison(
    data_2d: np.ndarray,
    true_labels: list,
    cluster_labels: np.ndarray,
    title_prefix: str,
    axes,
) -> None:
    """Side-by-side scatter: ground-truth labels vs. cluster assignments."""
    for ax, hue, suffix in [
        (axes[0], true_labels,    "Ground Truth"),
        (axes[1], cluster_labels, "Cluster Assignments"),
    ]:
        sns.scatterplot(
            x=data_2d[:, 0], y=data_2d[:, 1],
            hue=hue, palette="viridis", legend="full", ax=ax, s=15, alpha=0.7,
        )
        ax.set_title(f"{title_prefix} — {suffix}")
        ax.set_xlabel("PCA 1")
        ax.set_ylabel("PCA 2")


# Best TF-IDF: K-Means on raw matrix
tfidf_2d     = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(tfidf_matrix.toarray())
tfidf_cl     = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init="auto").fit_predict(tfidf_matrix)

# Best MiniLM: Agglomerative on SVD(50)
minilm_svd   = TruncatedSVD(n_components=50, random_state=RANDOM_STATE).fit_transform(minilm_embeds)
minilm_2d    = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(minilm_svd)
minilm_cl    = AgglomerativeClustering(n_clusters=2, linkage="ward").fit_predict(minilm_svd)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Part 1 — Review Length: Ground Truth vs. Discovered Clusters", fontsize=13)
plot_clustering_comparison(tfidf_2d,  gt_labels_p1, tfidf_cl,  "TF-IDF",  axes[0])
plot_clustering_comparison(minilm_2d, gt_labels_p1, minilm_cl, "MiniLM",  axes[1])
plt.tight_layout()
plt.savefig("part1_clustering_viz.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Part 2 — Unsupervised Game Genre Structure

Cluster **games** (not individual reviews) by what players praise in positive reviews.  
Each game is represented by a single aggregated vector.

- **TF-IDF game vector**: concatenate all positive reviews per game → one TF-IDF document  
- **MiniLM game vector**: embed each positive review → average embeddings per game  

**Task 3** profiles a held-out game by assigning it to the nearest cluster and discovering its complaint / praise themes via TF-IDF + Autoencoder + HDBSCAN.

### 2.1 — Build Per-Game Representations

In [ ]:
positive = df[df["recommend"] == True]

# Aggregate all positive reviews into one document per game
games_pos = (
    positive
    .groupby(["appid", "game_name", "release_date", "genres"])["processed_review_text"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

game_tfidf_vec    = TfidfVectorizer(min_df=3, stop_words="english", ngram_range=(1, 1))
game_tfidf_matrix = game_tfidf_vec.fit_transform(games_pos["processed_review_text"].astype(str))
print(f"TF-IDF game matrix    : {game_tfidf_matrix.shape}")

# Average MiniLM embeddings across all positive reviews per game
minilm_game    = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
ind_embeds     = minilm_game.encode(positive["review_text"].astype(str).tolist(), show_progress_bar=True)
pos_with_emb   = positive.copy()
pos_with_emb["emb"] = list(ind_embeds)

game_minilm_df = (
    pos_with_emb
    .groupby(["appid", "game_name", "release_date", "genres"])["emb"]
    .apply(lambda x: np.mean(list(x), axis=0))
    .reset_index()
)
game_minilm_matrix = np.vstack(game_minilm_df["emb"].values)
print(f"MiniLM game embeddings: {game_minilm_matrix.shape}")

### 2.2 — Game Clustering Benchmark (with Autoencoder)

In [ ]:
def build_autoencoder(input_dim: int, latent_dim: int = 50):
    """
    Build a shallow single-layer autoencoder.

    Parameters
    ----------
    input_dim  : int — dimensionality of input features
    latent_dim : int — bottleneck size (default 50)

    Returns
    -------
    (autoencoder, encoder) — compiled Keras models
    """
    inp     = Input(shape=(input_dim,))
    encoded = Dense(latent_dim, activation="relu")(inp)
    decoded = Dense(input_dim, activation="sigmoid")(encoded)
    ae      = Model(inputs=inp, outputs=decoded)
    enc     = Model(inputs=inp, outputs=encoded)
    ae.compile(optimizer="adam", loss="mse")
    return ae, enc


def top_genres_per_cluster(labels: np.ndarray, genres_series: pd.Series, n: int = 3) -> dict:
    """
    Return the top-n most frequent genre tags for each cluster.
    Handles comma-separated multi-label genre strings.
    """
    genres_series = genres_series.reset_index(drop=True)
    return {
        int(cid): [
            g for g, _ in Counter(
                g.strip()
                for gs in genres_series[labels == cid].dropna()
                for g in str(gs).split(",")
            ).most_common(n)
        ]
        for cid in np.unique(labels)
    }


def run_game_clustering(
    representations: dict,
    genres_series: pd.Series,
    n_clusters: int = 5,
    random_state: int = RANDOM_STATE,
) -> list:
    """
    Run game-level clustering across all pipeline combinations including Autoencoder DR.

    Returns
    -------
    list of dicts — one per successful pipeline with cluster sizes and top genres
    """
    results = []
    for rep_name, data in representations.items():
        print(f"\n>> {rep_name}")
        dense = data.toarray() if hasattr(data, "toarray") else data

        dr_configs = {
            "None"       : None,
            "SVD"        : TruncatedSVD(n_components=50, random_state=random_state),
            "UMAP"       : UMAP(n_components=50, random_state=random_state, n_neighbors=15, min_dist=0.1),
            "Autoencoder": "autoencoder",
        }
        cluster_configs = {
            "K-Means"      : KMeans(n_clusters=n_clusters, random_state=random_state, n_init="auto"),
            "Agglomerative": AgglomerativeClustering(n_clusters=n_clusters, linkage="ward"),
            "HDBSCAN"      : hdbscan.HDBSCAN(min_cluster_size=2),
        }

        for dr_name, dr_cfg in dr_configs.items():
            try:
                if dr_name == "None":
                    X = dense
                elif dr_name == "SVD":
                    X = dr_cfg.fit_transform(data)
                elif dr_name == "UMAP":
                    pre = TruncatedSVD(n_components=200, random_state=random_state).fit_transform(data) \
                          if rep_name == "TF-IDF" else data
                    X = dr_cfg.fit_transform(pre)
                elif dr_name == "Autoencoder":
                    ae, enc = build_autoencoder(dense.shape[1])
                    ae.fit(dense, dense, epochs=50, batch_size=32, verbose=0)
                    X = enc.predict(dense, verbose=0)
            except Exception as e:
                print(f"  DR failed [{dr_name}]: {e}")
                continue

            for cl_name, cl_model in cluster_configs.items():
                try:
                    labels     = cl_model.fit_predict(X)
                    n_found    = len(set(labels) - {-1})
                    noise_frac = (labels == -1).mean() if -1 in labels else 0.0
                    results.append({
                        "Representation"  : rep_name,
                        "DR"              : dr_name,
                        "Clustering"      : cl_name,
                        "N Clusters Found": n_found,
                        "Noise Fraction"  : round(noise_frac, 3),
                        "Cluster Sizes"   : dict(Counter(labels)),
                        "Top Genres"      : top_genres_per_cluster(labels, genres_series),
                    })
                except Exception as e:
                    print(f"  Cluster failed [{dr_name}+{cl_name}]: {e}")
    return results


game_results = run_game_clustering(
    representations={"TF-IDF": game_tfidf_matrix, "MiniLM": game_minilm_matrix},
    genres_series=games_pos["genres"],
)

summary_p2 = pd.DataFrame(
    [{k: v for k, v in r.items() if k not in ("Cluster Sizes", "Top Genres")} for r in game_results]
)
print("\nPart 2 — Summary:")
print(summary_p2.to_string(index=False))

### 2.3 — Multi-Genre Cluster Analysis (Best Pipeline: MiniLM + None + Agglomerative)

In [ ]:
def cluster_genre_purity(
    labels: np.ndarray,
    games_df: pd.DataFrame,
    embeddings: np.ndarray,
    n_representative: int = 2,
) -> list:
    """
    For each cluster, compute:
    - top 3 genres with frequency percentages
    - genre purity (fraction of games containing any top-3 genre)
    - representative games nearest to the cluster centroid

    Parameters
    ----------
    labels           : cluster assignment array
    games_df         : per-game DataFrame with 'genres' and 'game_name' columns
    embeddings       : game embedding matrix used for centroid distance
    n_representative : number of representative games to return per cluster
    """
    summary = []
    for cid in np.unique(labels):
        idx = np.where(labels == cid)[0]
        genre_strings = games_df.iloc[idx]["genres"].tolist()

        all_genres  = [g.strip() for gs in genre_strings for g in str(gs).split(",") if g.strip()]
        genre_counts = Counter(all_genres)
        total        = sum(genre_counts.values())

        top3 = [(g, c) for g, c in genre_counts.most_common(3)]
        top3_names = [g for g, _ in top3]
        top3_str   = [f"{g} ({100*c/total:.1f}%)" for g, c in top3]

        # Purity: fraction of games in cluster that contain at least one top-3 genre
        purity = sum(
            any(g in top3_names for g in str(gs).split(","))
            for gs in genre_strings
        ) / len(idx) * 100

        # Representative games: closest to cluster centroid
        centroid = embeddings[idx].mean(axis=0)
        dists    = np.linalg.norm(embeddings[idx] - centroid, axis=1)
        rep_idx  = idx[np.argsort(dists)[:n_representative]]
        reps     = [
            {"game": games_df.iloc[i]["game_name"], "genres": games_df.iloc[i]["genres"]}
            for i in rep_idx
        ]

        summary.append({
            "cluster"       : int(cid),
            "n_games"       : len(idx),
            "top_genres"    : top3_str,
            "purity_pct"    : round(purity, 1),
            "representatives": reps,
        })
    return summary


# Best pipeline: MiniLM + No DR + Agglomerative
best_labels = AgglomerativeClustering(n_clusters=5, linkage="ward").fit_predict(game_minilm_matrix)
cluster_analysis = cluster_genre_purity(best_labels, games_pos, game_minilm_matrix)

for c in cluster_analysis:
    print(f"\nCluster {c['cluster']} ({c['n_games']} games | Purity: {c['purity_pct']}%)")
    print(f"  Top genres : {', '.join(c['top_genres'])}")
    for r in c['representatives']:
        print(f"  Representative: {r['game']}  [{r['genres']}]")

### 2.4 — Held-Out Game Profiling (Genre Estimation + Theme Discovery)

In [ ]:
heldout = load_csv(COLAB_HELDOUT_PATH, LOCAL_HELDOUT_PATH, "held-out game")
heldout["processed_review_text"] = heldout["review_text"].astype(str).apply(preprocess)

# Build held-out game vector (average MiniLM over positive reviews)
heldout_pos = heldout[heldout["recommend"] == True]
heldout_embeds = minilm_game.encode(heldout_pos["review_text"].astype(str).tolist(), show_progress_bar=False)
heldout_vec    = heldout_embeds.mean(axis=0)  # shape: (384,)

# Assign to nearest cluster centroid
centroids = {
    cid: game_minilm_matrix[np.where(best_labels == cid)[0]].mean(axis=0)
    for cid in np.unique(best_labels)
}
assigned_cluster = min(centroids, key=lambda cid: np.linalg.norm(heldout_vec - centroids[cid]))

c = cluster_analysis[assigned_cluster]
print(f"Held-out game assigned to Cluster {assigned_cluster}")
print(f"  Estimated genres : {', '.join(c['top_genres'])}")
print(f"  Similar games    : {[r['game'] for r in c['representatives']]}")

In [ ]:
def autoencoder_hdbscan_cluster(data_matrix, latent_dim: int = 50, min_cluster_size: int = 5):
    """
    Reduce a sparse matrix with an autoencoder, then cluster with HDBSCAN.

    Returns
    -------
    (labels, latent_embeddings)
    """
    dense = data_matrix.toarray() if hasattr(data_matrix, "toarray") else data_matrix
    ae, enc = build_autoencoder(dense.shape[1], latent_dim)
    ae.fit(dense, dense, epochs=50, batch_size=32, verbose=0)
    latent = enc.predict(dense, verbose=0)
    labels = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size).fit_predict(latent)
    return labels, latent


def get_top_tfidf_terms(indices, matrix, vectorizer, n: int = 10) -> list:
    """Return the top-n TF-IDF terms for a cluster defined by row indices."""
    cluster_sum  = matrix[indices].sum(axis=0)
    dense_sum    = np.asarray(cluster_sum).flatten()
    feature_names = vectorizer.get_feature_names_out()
    top_idx      = dense_sum.argsort()[::-1][:n]
    return [f"{feature_names[i]} ({dense_sum[i]:.2f})" for i in top_idx]


def get_exemplar_reviews(indices, latent, reviews_df, n: int = 2) -> list:
    """Return the n reviews closest to the cluster centroid."""
    centroid = latent[indices].mean(axis=0)
    dists    = np.linalg.norm(latent[indices] - centroid, axis=1)
    closest  = [indices[i] for i in np.argsort(dists)[:n]]
    return reviews_df.iloc[closest]["review_text"].tolist()


complaints = heldout[heldout["recommend"] == False]
praises    = heldout[heldout["recommend"] == True]

complaints_tfidf_vec = TfidfVectorizer(min_df=3, stop_words="english")
complaints_matrix    = complaints_tfidf_vec.fit_transform(complaints["processed_review_text"])

praises_tfidf_vec    = TfidfVectorizer(min_df=3, stop_words="english")
praises_matrix       = praises_tfidf_vec.fit_transform(praises["processed_review_text"])

complaint_labels, complaint_latent = autoencoder_hdbscan_cluster(complaints_matrix)
praise_labels,    praise_latent    = autoencoder_hdbscan_cluster(praises_matrix)

print(f"Complaint clusters: {sorted(set(complaint_labels))}")
print(f"Praise clusters   : {sorted(set(praise_labels))}")

In [ ]:
def print_theme_clusters(labels, latent, matrix, vectorizer, reviews_df, theme_type: str) -> None:
    """Pretty-print top terms and exemplar reviews for each discovered theme cluster."""
    print(f"\n{'─'*60}")
    print(f"{theme_type.upper()} THEME CLUSTERS")
    print(f"{'─'*60}")

    noise_idx = np.where(labels == -1)[0]
    if len(noise_idx):
        print(f"\nNoise (-1): {len(noise_idx)} reviews ({100*len(noise_idx)/len(labels):.1f}%)")

    for cid in sorted(set(labels) - {-1}):
        idx = np.where(labels == cid)[0]
        if len(idx) < 5:
            continue
        top_terms = get_top_tfidf_terms(idx, matrix, vectorizer)
        exemplars = get_exemplar_reviews(idx, latent, reviews_df)

        print(f"\nCluster {cid} ({len(idx)} reviews)")
        print(f"  Top terms : {', '.join(top_terms[:6])}")
        for i, ex in enumerate(exemplars):
            print(f"  Exemplar {i+1}: {ex[:300]}...")


print_theme_clusters(complaint_labels, complaint_latent, complaints_matrix, complaints_tfidf_vec, complaints, "Complaints")
print_theme_clusters(praise_labels,    praise_latent,    praises_matrix,    praises_tfidf_vec,    praises,    "Praises")

### 2.5 — LLM Cluster Labelling

In [ ]:
LLM_MODEL = "Qwen/Qwen1.5-0.5B-Chat"

tokenizer_qwen = AutoTokenizer.from_pretrained(LLM_MODEL)
model_qwen     = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)
print(f"Loaded {LLM_MODEL} on {model_qwen.device}")


def generate_cluster_label(
    top_terms: list,
    exemplars: list,
    cluster_type: str,
    model,
    tokenizer,
    max_new_tokens: int = 20,
) -> str:
    """
    Use a small LLM to generate a 3–6 word descriptive label for a review cluster.

    Parameters
    ----------
    top_terms    : list of top TF-IDF term strings
    exemplars    : list of exemplar review texts
    cluster_type : 'complaint' or 'praise'
    model        : loaded HuggingFace causal LM
    tokenizer    : corresponding tokenizer
    """
    exemplar_block = "".join(f"    {i+1}. '{ex[:500]}...'\n" for i, ex in enumerate(exemplars))
    prompt = (
        f"You are an expert in text analysis. Analyze this cluster of {cluster_type} reviews.\n\n"
        f"Top TF-IDF terms: {', '.join(top_terms)}\n\n"
        f"Exemplar reviews:\n{exemplar_block}\n"
        f"Generate a concise, unique label STRICTLY 3–6 words long using only adjectives or "
        f"descriptive nouns. No quotes or prefixes.\n\nCluster Label:"
    )
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output = model.generate(
        inputs.input_ids, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9
    )
    return tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# Generate LLM labels for complaint and praise clusters
for theme_type, labels, latent, matrix, vec, reviews_df in [
    ("complaint", complaint_labels, complaint_latent, complaints_matrix, complaints_tfidf_vec, complaints),
    ("praise",    praise_labels,    praise_latent,    praises_matrix,    praises_tfidf_vec,    praises),
]:
    print(f"\n{'─'*50}")
    print(f"LLM Labels — {theme_type.title()} Clusters")
    print(f"{'─'*50}")
    for cid in sorted(set(labels) - {-1}):
        idx       = np.where(labels == cid)[0]
        if len(idx) < 5:
            continue
        terms     = get_top_tfidf_terms(idx, matrix, vec)
        exemplars = get_exemplar_reviews(idx, latent, reviews_df)
        llm_label = generate_cluster_label(terms, exemplars, theme_type, model_qwen, tokenizer_qwen)
        print(f"  Cluster {cid}: \"{llm_label}\"  (top terms: {', '.join(terms[:4])})")

---
# Part 3 — VGG16 Feature Extraction & Flower Clustering

Cluster the **TF-Flowers** dataset (5 flower classes) using features extracted from a
VGG16 network pre-trained on ImageNet. VGG16's early convolutional layers learn
transferable edge and texture detectors that remain discriminative for unseen datasets.

Feature extraction stops at the first fully-connected layer (4,096-dim dense vector per image).

### 3.1 — Load or Extract VGG16 Features

In [ ]:
class VGG16FeatureExtractor(nn.Module):
    """
    Truncated VGG16: outputs the 4096-dim activation from the first FC layer.
    Requires a GPU for practical speed (asserts CUDA availability).
    """

    def __init__(self):
        super().__init__()
        vgg         = torch.hub.load("pytorch/vision:v0.10.0", "vgg16", pretrained=True)
        self.features = nn.Sequential(*list(vgg.features))
        self.pooling  = vgg.avgpool
        self.flatten  = nn.Flatten()
        self.fc       = vgg.classifier[0]   # first FC layer → 4096-dim

    def forward(self, x):
        return self.fc(self.flatten(self.pooling(self.features(x))))


def load_or_extract_vgg_features(cache_path: str):
    """
    Load cached VGG16 features if available; otherwise download the flowers
    dataset and run extraction (requires GPU).

    Returns
    -------
    (f_all, y_all) — (N, 4096) float32 array, (N,) int array
    """
    if os.path.exists(cache_path):
        data = np.load(cache_path)
        print(f"Loaded cached features: {data['f_all'].shape}")
        return data["f_all"], data["y_all"]

    print("Cache not found — downloading flowers dataset and extracting features...")
    assert torch.cuda.is_available(), "GPU required for VGG16 feature extraction."

    if not os.path.exists("./flower_photos"):
        url = "http://download.tensorflow.org/example_images/flower_photos.tgz"
        with open("flower_photos.tgz", "wb") as f:
            f.write(requests.get(url).content)
        with tarfile.open("flower_photos.tgz") as t:
            t.extractall("./")
        os.remove("flower_photos.tgz")

    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    dataset    = datasets.ImageFolder(root="./flower_photos", transform=transform)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    extractor = VGG16FeatureExtractor().cuda().eval()
    f_all, y_all = np.zeros((0, 4096)), np.zeros((0,))
    for x, y in tqdm(dataloader, desc="Extracting features"):
        with torch.no_grad():
            f_all = np.vstack([f_all, extractor(x.cuda()).cpu().numpy()])
            y_all = np.concatenate([y_all, y.numpy()])

    np.savez(cache_path, f_all=f_all, y_all=y_all)
    print(f"Features extracted and cached: {f_all.shape}")
    return f_all, y_all


f_all, y_all = load_or_extract_vgg_features(VGG_FEATURES_PATH)
y_all        = y_all.astype(int)
print(f"Features: {f_all.shape} | Labels: {y_all.shape} | Classes: {np.unique(y_all)}")

### 3.2 — t-SNE Visualisation of VGG16 Features

In [ ]:
tsne     = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE)
f_tsne2d = tsne.fit_transform(f_all)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(f_tsne2d[:, 0], f_tsne2d[:, 1], c=y_all, cmap="viridis", s=15, alpha=0.8)
plt.colorbar(scatter, label="Flower class (ground truth)")
plt.title("t-SNE of VGG16 features — coloured by ground-truth label")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.tight_layout()
plt.savefig("part3_tsne_vgg.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.3 — Clustering Benchmark on VGG16 Features

In [ ]:
class DeepAutoencoder(torch.nn.Module, TransformerMixin):
    """
    Deep autoencoder for VGG16 features (4096 → 50).
    Implements fit / transform for sklearn pipeline compatibility.
    """

    def __init__(self, n_components: int = 50):
        super().__init__()
        self.n_components = n_components
        self.encoder = nn.Sequential(
            nn.Linear(4096, 1280), nn.ReLU(True),
            nn.Linear(1280, 640),  nn.ReLU(True),
            nn.Linear(640, 120),   nn.ReLU(True),
            nn.Linear(120, n_components),
        )
        self.decoder = nn.Sequential(
            nn.Linear(n_components, 120), nn.ReLU(True),
            nn.Linear(120, 640),          nn.ReLU(True),
            nn.Linear(640, 1280),         nn.ReLU(True),
            nn.Linear(1280, 4096),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def fit(self, X):
        X_t    = torch.tensor(X, dtype=torch.float32, device="cuda")
        self.cuda().train()
        opt    = torch.optim.Adam(self.parameters(), lr=1e-3, weight_decay=1e-5)
        loader = DataLoader(TensorDataset(X_t), batch_size=128, shuffle=True)
        for _ in tqdm(range(100), desc="Autoencoder"):
            for (xb,) in loader:
                loss = nn.MSELoss()(self(xb), xb)
                opt.zero_grad(); loss.backward(); opt.step()
        return self

    def transform(self, X) -> np.ndarray:
        self.eval()
        with torch.no_grad():
            return self.encoder(
                torch.tensor(X, dtype=torch.float32, device="cuda")
            ).cpu().numpy()


dr_models_img = {
    "None"       : None,
    "SVD"        : TruncatedSVD(n_components=50, random_state=RANDOM_STATE),
    "UMAP"       : UMAP(n_components=50, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1),
    "Autoencoder": DeepAutoencoder(n_components=50),
}
cluster_models_img = {
    "K-Means"      : KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init="auto"),
    "Agglomerative": AgglomerativeClustering(n_clusters=5, linkage="ward"),
    "HDBSCAN"      : hdbscan.HDBSCAN(min_cluster_size=10),
}

image_results = []
for dr_name, dr_model in dr_models_img.items():
    try:
        if dr_name == "None":
            X = f_all
        elif dr_name == "Autoencoder":
            X = dr_model.fit(f_all).transform(f_all)
        else:
            X = dr_model.fit_transform(f_all)
    except Exception as e:
        print(f"DR failed [{dr_name}]: {e}"); continue

    for cl_name, cl_model in cluster_models_img.items():
        try:
            labels     = cl_model.fit_predict(X)
            n_found    = len(set(labels) - {-1})
            noise_frac = (labels == -1).mean() if -1 in labels else 0.0
            image_results.append({
                "DR": dr_name, "Clustering": cl_name,
                "N Clusters": n_found, "Noise": round(noise_frac, 3),
                "ARI" : adjusted_rand_score(y_all, labels),
                "AMI" : adjusted_mutual_info_score(y_all, labels),
                "V"   : v_measure_score(y_all, labels),
            })
        except Exception as e:
            print(f"  Cluster failed [{dr_name}+{cl_name}]: {e}")

img_df = pd.DataFrame(image_results)
print("\nPart 3 — Image Clustering Results (sorted by ARI):")
print(img_df.sort_values("ARI", ascending=False).to_string(index=False))

### 3.4 — MLP Classifier on VGG16 Features

In [ ]:
class MLP(torch.nn.Module):
    """
    3-layer MLP classifier for 5-class flower type prediction.
    Operates on GPU; fit() and evaluate() manage tensor conversion internally.
    """

    def __init__(self, num_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 1280), nn.ReLU(True),
            nn.Linear(1280, 640),          nn.ReLU(True),
            nn.Linear(640, 5),             nn.LogSoftmax(dim=1),
        ).cuda()

    def forward(self, x):
        return self.net(x)

    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 100):
        """Train the classifier with Adam + NLLLoss."""
        X_t = torch.tensor(X, dtype=torch.float32, device="cuda")
        y_t = torch.tensor(y, dtype=torch.int64,   device="cuda")
        opt = torch.optim.Adam(self.parameters(), lr=1e-3, weight_decay=1e-5)
        loader = DataLoader(TensorDataset(X_t, y_t), batch_size=128, shuffle=True)
        self.train()
        for _ in tqdm(range(epochs), desc="MLP train"):
            for xb, yb in loader:
                loss = nn.NLLLoss()(self(xb), yb)
                opt.zero_grad(); loss.backward(); opt.step()
        return self

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return accuracy on a held-out test set."""
        self.eval()
        X_t = torch.tensor(X, dtype=torch.float32, device="cuda")
        y_t = torch.tensor(y, dtype=torch.int64,   device="cuda")
        with torch.no_grad():
            preds = self(X_t).argmax(dim=1)
        acc = (preds == y_t).float().mean().item()
        print(f"Accuracy: {acc*100:.2f}%")
        return acc


f_train, f_test, y_train, y_test = train_test_split(f_all, y_all, test_size=0.2, random_state=RANDOM_STATE)

# Original 4096-dim features
mlp_orig = MLP(f_train.shape[1]).fit(f_train, y_train)
print("Accuracy — original VGG features:")
acc_orig = mlp_orig.evaluate(f_test, y_test)

# UMAP-reduced 50-dim features
umap_dr     = UMAP(n_components=50, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
f_train_umap = umap_dr.fit_transform(f_train)
f_test_umap  = umap_dr.transform(f_test)

mlp_umap = MLP(f_train_umap.shape[1]).fit(f_train_umap, y_train)
print("Accuracy — UMAP-reduced features:")
acc_umap = mlp_umap.evaluate(f_test_umap, y_test)

print(f"\nAccuracy drop from DR: {(acc_orig - acc_umap)*100:.1f}pp")

---
# Part 4 — Pokémon CLIP Multimodal Retrieval & Type Classification

CLIP (Contrastive Language–Image Pretraining) maps images and text into a shared embedding
space. We use it to:
1. **Retrieve** Pokémon images matching type-based text queries
2. **Classify** each Pokémon's primary type from its image alone
3. **Re-rank** CLIP's top-5 candidates using a VLM (Qwen3-VL) to improve Acc@1

### 4.1 — Setup & Data Loading

In [ ]:
def load_clip_model(model_name: str = "ViT-L/14"):
    """Load CLIP model and preprocessing transform; auto-selects GPU if available."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, preprocess = clip.load(model_name, device=device)
    return model, preprocess, device


def construct_pokedex(csv_path: str, image_dir: str, type_filter: list = None) -> pd.DataFrame:
    """
    Build a Pokédex DataFrame with image paths.

    Parameters
    ----------
    csv_path     : path to Pokemon.csv
    image_dir    : directory containing per-Pokémon image folders (Name/0.jpg)
    type_filter  : optional list of Type1 values to restrict the dataset

    Returns
    -------
    pd.DataFrame — one row per Pokémon with a valid image path
    """
    pokedex = pd.read_csv(csv_path)
    pokedex["image_path"] = [
        next(iter(glob(f"{image_dir}/{name}/0.jpg")), None)
        for name in pokedex["Name"]
    ]
    pokedex = pokedex[pokedex["image_path"].notna()].copy()

    # Keep only Pokémon with unique IDs (removes form duplicates)
    unique_ids = pokedex["ID"].value_counts()
    pokedex    = pokedex[pokedex["ID"].isin(unique_ids[unique_ids == 1].index)].reset_index(drop=True)
    pokedex["Type2"] = pokedex["Type2"].astype(str).str.strip()

    if type_filter:
        pokedex = pokedex[pokedex["Type1"].isin(type_filter)].reset_index(drop=True)

    return pokedex


def encode_images(model, preprocess, image_paths: list, device: str) -> np.ndarray:
    """Run CLIP image encoder; returns L2-normalised embeddings."""
    embeddings = []
    with torch.no_grad():
        for path in tqdm(image_paths, desc="Image encoding"):
            img = preprocess(Image.open(path)).unsqueeze(0).to(device)
            embeddings.append(model.encode_image(img).cpu().numpy())
    E = np.concatenate(embeddings, axis=0)
    return E / np.linalg.norm(E, axis=1, keepdims=True)


def encode_texts(model, texts: list, device: str) -> np.ndarray:
    """Run CLIP text encoder; returns L2-normalised embeddings."""
    with torch.no_grad():
        E = model.encode_text(clip.tokenize(texts).to(device)).cpu().numpy()
    return E / np.linalg.norm(E, axis=1, keepdims=True)


# Download dataset and Pokédex CSV
dataset_path  = kagglehub.dataset_download("hlrhegemony/pokemon-image-dataset")
image_dir     = f"{dataset_path}/images"

pokemon_csv   = "Pokemon.csv"
if not os.path.exists(pokemon_csv):
    r = requests.get(POKEMON_CSV_URL)
    with open(pokemon_csv, "wb") as f:
        f.write(r.content)

clip_model, clip_preprocess, clip_device = load_clip_model()
pokedex = construct_pokedex(pokemon_csv, image_dir)
print(f"Pokédex: {len(pokedex)} entries | Device: {clip_device}")

image_embeddings = encode_images(clip_model, clip_preprocess, pokedex["image_path"].tolist(), clip_device)
print(f"Image embeddings: {image_embeddings.shape}")

### 4.2 — Text Query → Image Retrieval

In [ ]:
# Queries tuned via experimentation — more specific prompts improve retrieval for ambiguous types
TYPE_QUERIES = {
    "Bug"   : "a photo of a bug type pokemon",
    "Fire"  : "a photo of a fire type pokemon",
    "Grass" : "a photo of a grass type pokemon",
    "Dark"  : "a photo of a heavy all black dark type pokemon with fur and claws, a monster",
    "Dragon": "a photo of a powerful draconic dragon type pokemon with scales and monstrous reptilian features. not fighting type",
}


def plot_top5_retrieval(query_name: str, query_text: str, image_embeds: np.ndarray, df: pd.DataFrame) -> None:
    """Retrieve and plot the top-5 Pokémon images for a text query."""
    text_emb   = encode_texts(clip_model, [query_text], clip_device)
    scores     = softmax(100.0 * image_embeds @ text_emb.T, axis=0).flatten()
    top5_idx   = np.argsort(scores)[::-1][:5]

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    fig.suptitle(f"Query: '{query_text}'", fontsize=13)
    for ax, idx in zip(axes, top5_idx):
        p     = df.iloc[idx]
        types = f"{p['Type1']}, {p['Type2']}" if p["Type2"] not in ("", "nan") else p["Type1"]
        ax.imshow(Image.open(p["image_path"]))
        ax.set_title(f"{p['Name']}\n({types})", fontsize=9)
        ax.axis("off")
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(f"clip_retrieval_{query_name.lower()}.png", dpi=120, bbox_inches="tight")
    plt.show()


for name, query in TYPE_QUERIES.items():
    plot_top5_retrieval(name, query, image_embeddings, pokedex)

### 4.3 — Zero-Shot Type Classification (Acc@1 & Hit@5)

In [ ]:
all_types     = sorted(set(pokedex["Type1"]) | set(pokedex["Type2"].replace("nan", np.nan).dropna()))
type_queries  = [f"a photo of a {t} type pokemon" for t in all_types]
type_embeds   = encode_texts(clip_model, type_queries, clip_device)

# Similarity matrix: (N_pokemon, N_types)
all_sims = softmax(100.0 * image_embeddings @ type_embeds.T, axis=-1)

acc1_hits, hit5_hits = 0, 0
for i, gt_type in enumerate(pokedex["Type1"]):
    top5 = [all_types[j] for j in np.argsort(all_sims[i])[::-1][:5]]
    acc1_hits += top5[0] == gt_type
    hit5_hits += gt_type in top5

acc1 = acc1_hits / len(pokedex)
hit5 = hit5_hits / len(pokedex)
print(f"CLIP Acc@1  : {acc1:.4f}")
print(f"CLIP Hit@5  : {hit5:.4f}")

### 4.4 — VLM Re-ranking (Qwen3-VL)

In [ ]:
def load_qwen3_vl(model_id: str = "Qwen/Qwen3-VL-2B-Instruct"):
    """Load Qwen3-VL model and processor onto available device."""
    model     = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id, torch_dtype="auto", device_map="auto"
    ).eval()
    processor = AutoProcessor.from_pretrained(model_id)
    return model, processor


@torch.no_grad()
def qwen_vl_infer(model, processor, image_path: str, prompt: str, max_new_tokens: int = 128) -> str:
    """
    Run Qwen3-VL on a single image + text prompt.

    Parameters
    ----------
    image_path     : local file path to the image (do NOT use file:// URI)
    prompt         : instruction text
    max_new_tokens : generation budget

    Returns
    -------
    str — generated text with prompt tokens stripped
    """
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text",  "text": prompt},
        ],
    }]
    inputs     = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed    = [o[len(i):] for i, o in zip(inputs.input_ids, output_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()


vlm_model, vlm_processor = load_qwen3_vl()
vlm_processor.image_processor.min_pixels = vlm_processor.image_processor.max_pixels = 224 * 224
print("Qwen3-VL loaded.")

In [ ]:
subset_df = pokedex.sample(frac=0.5, random_state=RANDOM_STATE)
reranked_hits, parsing_errors, disagreements = 0, 0, 0

for idx, pokemon in tqdm(subset_df.iterrows(), total=len(subset_df), desc="VLM re-ranking"):
    gt_type      = pokemon["Type1"]
    top5_types   = [all_types[j] for j in np.argsort(all_sims[idx])[::-1][:5]]
    clip_top1    = top5_types[0]

    candidates   = list(top5_types)
    random.shuffle(candidates)
    prompt       = f"Options: {','.join(candidates)}. Based on the image, pick the best type from the options. Respond with ONLY one word."

    try:
        raw = qwen_vl_infer(vlm_model, vlm_processor, pokemon["image_path"], prompt)
        vlm_choice = next(
            (c for c in top5_types if c.upper() in raw.upper()),
            None
        )
    except Exception:
        vlm_choice = None

    if vlm_choice is None:
        parsing_errors += 1
    elif vlm_choice != clip_top1:
        disagreements += 1

    final = vlm_choice or clip_top1
    reranked_hits += final == gt_type

n = len(subset_df)
vlm_acc1 = reranked_hits / n

print("\n" + "="*50)
print(f"{'Metric':<30} | {'Value':>10}")
print("-"*50)
print(f"{'CLIP Acc@1':<30} | {acc1:>10.4f}")
print(f"{'CLIP Hit@5':<30} | {hit5:>10.4f}")
print(f"{'VLM Re-ranked Acc@1':<30} | {vlm_acc1:>10.4f}")
print(f"{'Parsing Error Rate':<30} | {parsing_errors/n:>10.4f}")
print(f"{'Total Disagreements':<30} | {disagreements:>10}")
print("="*50)

---
# Part 5 — SQL Analytics on Steam Reviews

The same Steam dataset loaded into a normalised **SQLite** schema for relational analysis.  
All aggregations below are done entirely in SQL — no pandas post-processing after the query.

**Schema:**
- `reviews` — one row per review with a derived `review_length` column
- `games` — deduplicated game metadata
- `game_genres` — normalised bridge table (`appid × genre`)

### 5.1 — Build SQLite Database

In [ ]:
def build_database(source_df: pd.DataFrame, db_path: str) -> sqlite3.Connection:
    """
    Ingest a Steam reviews DataFrame into a normalised SQLite database.

    Tables created
    --------------
    reviews     — one row per review, includes derived review_length (word count)
    games       — deduplicated game metadata (appid, game_name, release_date)
    game_genres — bridge table: one row per (appid, genre) pair
    """
    conn = sqlite3.connect(db_path)

    reviews = source_df[[
        "user", "playtime", "post_date", "helpfulness",
        "review_text", "recommend", "early_access_review",
        "appid", "game_name", "release_date",
    ]].copy()
    reviews["review_length"] = reviews["review_text"].astype(str).apply(lambda x: len(x.split()))
    reviews["recommend"]     = reviews["recommend"].astype(int)
    reviews.to_sql("reviews", conn, if_exists="replace", index_label="review_id")

    games = (
        source_df[["appid", "game_name", "release_date", "genres"]]
        .drop_duplicates(subset="appid")
        .reset_index(drop=True)
    )
    games.to_sql("games", conn, if_exists="replace", index=False)

    genre_rows = [
        {"appid": row["appid"], "genre": g.strip()}
        for _, row in games.iterrows()
        if pd.notna(row["genres"])
        for g in str(row["genres"]).split(",")
        if g.strip()
    ]
    pd.DataFrame(genre_rows).to_sql("game_genres", conn, if_exists="replace", index=False)

    conn.commit()
    print(f"reviews: {len(reviews):,} rows | games: {len(games):,} | genres: {len(genre_rows):,}")
    return conn


def sql(conn: sqlite3.Connection, query: str) -> pd.DataFrame:
    """Execute a SQL query and return results as a DataFrame."""
    return pd.read_sql_query(query, conn)


con = build_database(df, DB_PATH)
print("Database ready.")

### 5.2 — EDA Queries

In [ ]:
# Overall dataset stats
sql(con, """
    SELECT
        COUNT(*)                                           AS total_reviews,
        ROUND(100.0 * SUM(recommend) / COUNT(*), 2)       AS pct_positive,
        ROUND(AVG(review_length), 1)                       AS avg_words,
        ROUND(AVG(helpfulness), 3)                         AS avg_helpfulness
    FROM reviews
""")

In [ ]:
# Top 10 games by volume
sql(con, """
    SELECT game_name,
           COUNT(*)                                            AS reviews,
           ROUND(100.0 * SUM(recommend) / COUNT(*), 1)         AS pct_positive,
           ROUND(AVG(review_length), 1)                         AS avg_words
    FROM reviews
    GROUP BY game_name
    ORDER BY reviews DESC
    LIMIT 10
""")

### 5.3 — Genre Analysis (JOIN)

In [ ]:
# Sentiment and playtime by genre (JOIN reviews → game_genres)
sql(con, """
    SELECT
        gg.genre,
        COUNT(r.review_id)                                    AS reviews,
        ROUND(100.0 * SUM(r.recommend) / COUNT(*), 1)         AS pct_positive,
        ROUND(AVG(r.playtime), 1)                             AS avg_playtime_hrs,
        ROUND(AVG(r.helpfulness), 3)                          AS avg_helpfulness
    FROM reviews r
    JOIN game_genres gg ON r.appid = gg.appid
    GROUP BY gg.genre
    HAVING reviews >= 50
    ORDER BY pct_positive DESC
    LIMIT 15
""")

### 5.4 — Review Length Bands (NTILE Window Function)

In [ ]:
# Replicate Part 1 pseudo-label logic entirely in SQL
sql(con, """
    WITH quartile_labels AS (
        SELECT
            recommend,
            helpfulness,
            review_length,
            NTILE(4) OVER (ORDER BY review_length) AS quartile
        FROM reviews
    )
    SELECT
        CASE quartile
            WHEN 1 THEN 'Q1 Short'
            WHEN 2 THEN 'Q2'
            WHEN 3 THEN 'Q3'
            WHEN 4 THEN 'Q4 Long'
        END                                             AS length_band,
        COUNT(*)                                        AS reviews,
        ROUND(MIN(review_length))                       AS min_words,
        ROUND(MAX(review_length))                       AS max_words,
        ROUND(AVG(review_length), 1)                    AS avg_words,
        ROUND(100.0 * AVG(recommend), 1)                AS pct_positive,
        ROUND(AVG(helpfulness), 3)                      AS avg_helpfulness
    FROM quartile_labels
    GROUP BY quartile
    ORDER BY quartile
""")

### 5.5 — Window Function Showcase

In [ ]:
# Best-reviewed game per genre using RANK() with PARTITION BY
sql(con, """
    WITH game_stats AS (
        SELECT
            r.game_name,
            gg.genre,
            COUNT(*)                                          AS reviews,
            ROUND(100.0 * SUM(r.recommend) / COUNT(*), 1)     AS pct_positive
        FROM reviews r
        JOIN game_genres gg ON r.appid = gg.appid
        GROUP BY r.game_name, gg.genre
        HAVING reviews >= 20
    ),
    ranked AS (
        SELECT *,
               RANK() OVER (PARTITION BY genre ORDER BY pct_positive DESC) AS genre_rank
        FROM game_stats
    )
    SELECT genre, game_name, reviews, pct_positive
    FROM ranked
    WHERE genre_rank = 1
    ORDER BY genre
""")

In [ ]:
# Review length deciles: helpfulness trend across deciles
sql(con, """
    WITH d AS (
        SELECT helpfulness, recommend, review_length,
               NTILE(10) OVER (ORDER BY review_length) AS decile
        FROM reviews
    )
    SELECT decile,
           COUNT(*)                          AS reviews,
           ROUND(MIN(review_length))         AS min_words,
           ROUND(MAX(review_length))         AS max_words,
           ROUND(AVG(helpfulness), 3)        AS avg_helpfulness,
           ROUND(100.0 * AVG(recommend), 1)  AS pct_positive
    FROM d
    GROUP BY decile
    ORDER BY decile
""")

In [ ]:
# CTE: multi-genre games — do more genre tags correlate with more reviews?
sql(con, """
    WITH genre_counts AS (
        SELECT appid, COUNT(*) AS num_genres FROM game_genres GROUP BY appid
    ),
    review_counts AS (
        SELECT appid, COUNT(*) AS num_reviews FROM reviews GROUP BY appid
    )
    SELECT
        gc.num_genres,
        COUNT(rc.appid)               AS num_games,
        ROUND(AVG(rc.num_reviews), 1)  AS avg_reviews_per_game
    FROM genre_counts gc
    JOIN review_counts rc ON gc.appid = rc.appid
    GROUP BY gc.num_genres
    ORDER BY gc.num_genres
""")

### 5.6 — Visualise SQL Results

In [ ]:
# Genre recommendation rate bar chart
genre_sentiment = sql(con, """
    SELECT gg.genre,
           ROUND(100.0 * SUM(r.recommend) / COUNT(*), 1) AS pct_positive,
           COUNT(*) AS reviews
    FROM reviews r
    JOIN game_genres gg ON r.appid = gg.appid
    GROUP BY gg.genre
    HAVING reviews >= 100
    ORDER BY pct_positive DESC
    LIMIT 15
""")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=genre_sentiment, x="pct_positive", y="genre", palette="viridis", ax=axes[0])
axes[0].set_xlabel("% Positive Reviews")
axes[0].set_title("Recommendation Rate by Genre (min 100 reviews)")

decile_df = sql(con, """
    WITH d AS (SELECT helpfulness, review_length, NTILE(10) OVER (ORDER BY review_length) AS decile FROM reviews)
    SELECT decile, ROUND(AVG(review_length)) AS avg_words, ROUND(AVG(helpfulness), 3) AS avg_helpfulness
    FROM d GROUP BY decile ORDER BY decile
""")
ax2 = axes[1].twinx()
axes[1].bar(decile_df["decile"], decile_df["avg_words"], color="steelblue", alpha=0.5, label="Avg Words")
ax2.plot(decile_df["decile"], decile_df["avg_helpfulness"], color="coral", marker="o", label="Avg Helpfulness")
axes[1].set_xlabel("Review Length Decile (1=shortest)")
axes[1].set_ylabel("Avg Word Count", color="steelblue")
ax2.set_ylabel("Avg Helpfulness", color="coral")
axes[1].set_title("Review Length vs. Helpfulness by Decile")

plt.tight_layout()
plt.savefig("part5_sql_viz.png", dpi=150, bbox_inches="tight")
plt.show()

con.close()
print("Database connection closed.")